# 3. XGBoost — Predicting Electric Vehicle Purchases

Primer modelo de árboles del episodio. El baseline lineal de `2_logit.ipynb` llegó a
**CV AUC = 0,93855** (score público 0,93789), y su conclusión fue que lo que faltaba no eran
transformaciones evidentes sino **interacciones de orden mayor** entre las variables fuertes. Este
notebook pone esa hipótesis a prueba.

Conviene ser preciso sobre qué aporta XGBoost **en este dataset**, porque no es lo mismo que en el
Episodio 8:

- **Los NaN nativos no suman nada acá.** El EDA confirmó cero nulos en train y en test, así que la
  ventaja que en el Ep. 8 fue decisiva (allá había entre 4 % y 19 % de faltantes en todas las
  features) acá es irrelevante.
- **Las categóricas nativas sí importan.** `Gender`, `City_Type` y `Current_Car_Type` son nominales;
  con `enable_categorical=True` se pasan como dtype `category` y el algoritmo agrupa niveles por su
  cuenta, sin imponerles el orden falso de un `OrdinalEncoder` ni pagar el costo de un one-hot.
- **Las interacciones son la ventaja real.** El logit necesitaba que le construyéramos a mano cada
  cruce; un ensamble de árboles las arma solo, y a cualquier profundidad.

**Validación:** `StratifiedKFold(5)` con *early stopping* por fold, predicciones out-of-fold reales y
predicción de test promediada entre los 5 modelos.

> **Tiempo de ejecución: ~45 minutos.** Corre en CPU (`tree_method="hist"`): este equipo no tiene GPU
> NVIDIA y son 4 núcleos físicos. Referencia medida en una prueba de un fold: **266 s por fold**
> (early stopping en 499 árboles, AUC 0,94069), o sea ~22 min por configuración.

**Métrica:** ROC AUC — **Dataset:** Playground Series S6E9

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["figure.dpi"] = 100

SEED    = 42
N_FOLDS = 5

## 1. Carga y Preprocesamiento

In [ ]:
train_raw = pd.read_csv("../data/train.csv")
test_raw  = pd.read_csv("../data/test.csv")

TARGET = "Will_Buy_EV"

crudas_num = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
              "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
              "Environmental_Concern_Level"]
crudas_cat = ["Gender", "City_Type", "Current_Car_Type",
              "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]


def agregar_derivadas(df):
    """Solo las derivadas de 2_logit.ipynb que un arbol NO puede construir por su cuenta."""
    d = df.copy()
    d["income_per_car"] = d["Annual_Income_USD"] / d["Number_of_Cars_Owned"]
    d["charging_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d.replace([np.inf, -np.inf], np.nan)


derivadas = ["income_per_car", "charging_total"]

train = agregar_derivadas(train_raw)
test  = agregar_derivadas(test_raw)
y     = (train[TARGET] == "Yes").astype(int).values

print(f"Train: {train_raw.shape[0]:,} filas x {train_raw.shape[1]} columnas")
print(f"Test:  {test_raw.shape[0]:,} filas x {test_raw.shape[1]} columnas")
print(f"Tasa base: {y.mean():.4%}")
print(f"\nDerivadas ({len(derivadas)}): {derivadas}")

**Por qué el logit llevaba tres derivadas y acá quedan dos.** `log_income` se descarta a propósito:
un árbol parte por umbrales (`ingreso < 67.376`), y cualquier transformación **monótona** produce
exactamente los mismos puntos de corte y las mismas particiones. Para un modelo lineal el logaritmo
cambiaba la forma de la relación y aportaba; para un árbol es una columna redundante que sólo agrega
ruido al muestreo de `colsample_bytree`.

`income_per_car` (un cociente) y `charging_total` (una suma) sí son features nuevas: un árbol las
aproximaría con muchos splits sucesivos, o directamente no las encontraría.

Lo mismo con las otras derivadas del logit: `env_concern_cat` no tiene sentido acá (el árbol ya parte
un ordinal donde quiera) y `home_x_city` es justamente el tipo de interacción que XGBoost construye
solo — dárselo hecho sería redundante.

In [ ]:
# Dtype categorico con vocabulario compartido entre train y test, para que los codigos internos
# coincidan en ambos splits.
for col in crudas_cat:
    niveles = sorted(set(train[col].dropna()) | set(test[col].dropna()))
    dtype   = pd.CategoricalDtype(categories=niveles, ordered=False)
    train[col] = train[col].astype(dtype)
    test[col]  = test[col].astype(dtype)
    print(f"  {col:24s} -> {niveles}")

n_tr = train[crudas_num + derivadas + crudas_cat].isnull().sum().sum()
n_te = test[crudas_num + derivadas + crudas_cat].isnull().sum().sum()
print(f"\nNulos en las features (XGBoost los manejaria nativamente, pero no hay):")
print(f"  train: {n_tr}  |  test: {n_te}")

## 2. Configuración del Modelo

Parámetros conservadores, sin búsqueda de hiperparámetros: el techo de 3000 árboles es alto a
propósito y quien decide cuántos se usan realmente es el **early stopping** contra el fold de
validación. Así cada fold corta donde le conviene y no hace falta fijar `n_estimators` a mano.

In [ ]:
PARAMS = dict(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=3000,          # techo; el early stopping corta mucho antes
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",         # CPU: este equipo no tiene GPU NVIDIA
    enable_categorical=True,
    early_stopping_rounds=100,
    random_state=SEED,
    n_jobs=-1,
)


def cv_xgboost(X, y, X_test, params, skf, etiqueta=""):
    """5-fold CV con early stopping. Devuelve OOF real y test promediado entre folds."""
    oof      = np.zeros(len(X), dtype=np.float32)
    test_sum = np.zeros(len(X_test), dtype=np.float32)
    aucs, iters, modelos = [], [], []

    print(f"{'='*66}")
    print(f"CROSS-VALIDATION — {etiqueta}")
    print(f"{'='*66}")

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        modelo = XGBClassifier(**params)
        modelo.fit(X.iloc[tr_idx], y[tr_idx],
                   eval_set=[(X.iloc[val_idx], y[val_idx])], verbose=False)

        p_val         = modelo.predict_proba(X.iloc[val_idx])[:, 1]
        oof[val_idx]  = p_val
        test_sum     += modelo.predict_proba(X_test)[:, 1]

        auc = roc_auc_score(y[val_idx], p_val)
        aucs.append(auc)
        iters.append(modelo.best_iteration)
        modelos.append(modelo)
        print(f"  Fold {fold}: AUC = {auc:.5f}  |  best_iteration = {modelo.best_iteration}")

    print(f"{'='*66}")
    print(f"  Media: {np.mean(aucs):.5f} +/- {np.std(aucs):.5f}")
    print(f"  AUC out-of-fold: {roc_auc_score(y, oof):.5f}")
    print(f"  Arboles usados (media): {np.mean(iters):.0f}")
    print(f"{'='*66}\n")

    return dict(oof=oof, test=test_sum / skf.get_n_splits(),
                aucs=aucs, iters=iters, modelos=modelos)

## 3. Validación Cross-Validation

Se comparan las mismas dos configuraciones que en el logit. Allá las derivadas sumaban apenas
+0,00045, y la explicación era que la relación ya era casi lineal en log-odds. Acá la pregunta es
distinta: si un árbol construye interacciones solo, ¿le sigue sirviendo que le demos un cociente y
una suma hechos?

> Esta es la celda pesada: ~45 minutos entre las dos configuraciones.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

configs = {
    "B. crudas":             crudas_num + crudas_cat,
    "C. crudas + derivadas": crudas_num + derivadas + crudas_cat,
}

resultados = {}
for nombre, feats in configs.items():
    resultados[nombre] = cv_xgboost(train[feats], y, test[feats], PARAMS, skf, etiqueta=nombre)
    resultados[nombre]["features"] = feats

MEJOR    = max(resultados, key=lambda k: np.mean(resultados[k]["aucs"]))
res      = resultados[MEJOR]
features = res["features"]
cv_mean  = np.mean(res["aucs"])

print(f"{'='*66}")
print("COMPARATIVA DE CONFIGURACIONES")
print(f"{'='*66}")
for nombre, r in resultados.items():
    print(f"  {nombre:24s} AUC = {np.mean(r['aucs']):.5f} +/- {np.std(r['aucs']):.5f}")
print(f"{'='*66}")
print(f"  Mejor: {MEJOR}  (AUC = {cv_mean:.5f})")

## 4. Diagnóstico — Curva ROC y Separación de Probabilidades

In [ ]:
oof     = res["oof"]
auc_oof = roc_auc_score(y, oof)
print(f"AUC out-of-fold ({MEJOR}): {auc_oof:.5f}")

fpr, tpr, _ = roc_curve(y, oof)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].plot(fpr, tpr, color="tomato", lw=2, label=f"xgboost (AUC = {auc_oof:.4f})")
axes[0].plot([0, 1], [0, 1], color="gray", ls=":", lw=1.5, label="azar")
axes[0].set_xlabel("Tasa de falsos positivos")
axes[0].set_ylabel("Tasa de verdaderos positivos")
axes[0].set_title("Curva ROC (predicciones out-of-fold)")
axes[0].legend(loc="lower right")
for cls, color, etiqueta in [(0, "steelblue", "No"), (1, "tomato", "Yes")]:
    axes[1].hist(oof[y == cls], bins=60, alpha=0.6, color=color,
                 label=f"Will_Buy_EV = {etiqueta}")
axes[1].set_xlabel("Probabilidad predicha")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Separacion de las probabilidades out-of-fold")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Feature Importance

Con early stopping cada fold corta en un `best_iteration` distinto, así que no hay un único "modelo
final": la importancia se promedia entre los 5 modelos y se muestra el desvío como barra de error.

In [ ]:
imps      = pd.DataFrame([m.feature_importances_ for m in res["modelos"]], columns=features)
imp_media = imps.mean().sort_values()
imp_std   = imps.std().reindex(imp_media.index)

fig, ax = plt.subplots(figsize=(11, max(6, len(features) * 0.42)))
ax.barh(imp_media.index, imp_media.values, xerr=imp_std.values,
        color="steelblue", edgecolor="black", error_kw=dict(ecolor="gray", lw=1))
ax.set_xlabel("Importancia (gain), promedio de los 5 folds")
ax.set_title("Feature Importance — XGBoost", fontsize=14)
plt.tight_layout()
plt.show()

print("Ordenada por importancia:")
print(imp_media.sort_values(ascending=False).round(4).to_string())

## 6. Comparación con el Baseline Lineal

El logit dejó dos números para contrastar: **CV AUC 0,93855** y **score público 0,93789**. La brecha
entre ambos (la CV quedó 0,00066 por encima del leaderboard) sirve como referencia de cuánto se puede
esperar que se mueva el score al enviar.

In [ ]:
AUC_LOGIT_CV = 0.93855   # CV AUC de 2_logit.ipynb (crudas + derivadas)
AUC_LOGIT_LB = 0.93789   # score publico de ese mismo envio
TOPE_LB      = 0.94631   # primer puesto del leaderboard al momento de escribir esto

comparativa = pd.DataFrame({
    "modelo": ["Logit (2_logit.ipynb)", f"XGBoost ({MEJOR})"],
    "CV AUC": [AUC_LOGIT_CV, cv_mean],
})
comparativa["delta vs logit"] = (comparativa["CV AUC"] - AUC_LOGIT_CV).round(5)
comparativa["brecha vs tope"] = (TOPE_LB - comparativa["CV AUC"]).round(5)

print(f"{'='*66}")
print("COMPARATIVA DE MODELOS")
print(f"{'='*66}")
print(comparativa.round(5).to_string(index=False))
print(f"{'='*66}")
print(f"  Ganancia de XGBoost sobre el logit: {cv_mean - AUC_LOGIT_CV:+.5f}")
print(f"  Desvio entre folds de XGBoost:      {np.std(res['aucs']):.5f}")

In [ ]:
# Correlacion entre los dos modelos: cuanto de nuevo aporta XGBoost sobre el logit.
# Si fuera muy alta, un ensamble de los dos no tendria mucho sentido.
oof_logit = np.load("../models/2_logit_oof.npy")
print(f"Correlacion de Pearson   OOF logit vs OOF xgboost: {np.corrcoef(oof_logit, oof)[0, 1]:.4f}")
print(f"Correlacion de Spearman (la que importa para el AUC): "
      f"{pd.Series(oof_logit).corr(pd.Series(oof), method='spearman'):.4f}")

## 7. Predicción sobre Test y Submission

In [ ]:
proba_test = res["test"]

submission = pd.DataFrame({"id": test["id"], TARGET: proba_test})
submission.to_csv("../submissions/3_xgboost_submission.csv", index=False)

# OOF y predicciones de test para un ensamble futuro (models/ esta gitignoreado)
np.save("../models/3_xgboost_oof.npy",  oof)
np.save("../models/3_xgboost_test.npy", proba_test)

print(f"Submission escrita: ../submissions/3_xgboost_submission.csv ({len(submission):,} filas)")
print(submission.head().to_string(index=False))
print(f"\nProbabilidad media predicha: {proba_test.mean():.4f}"
      f"  (tasa base en train: {y.mean():.4f})")

In [ ]:
# Chequeo de sanidad contra sample_submission (formato e ids)
sample = pd.read_csv("../data/sample_submission.csv")
assert list(submission.columns) == list(sample.columns), "columnas no coinciden"
assert len(submission) == len(sample), "cantidad de filas no coincide"
assert (submission["id"].values == sample["id"].values).all(), "los ids no estan alineados"
assert submission[TARGET].between(0, 1).all(), "hay probabilidades fuera de [0, 1]"
assert submission[TARGET].notna().all(), "hay nulos en la prediccion"
print("Chequeos OK — submission lista para enviar.")

## 8. Envío a Kaggle vía API

En este equipo Avast intercepta TLS e inyecta su propia root CA, que no está en el bundle de
`certifi` — sin el fix de abajo la CLI falla con `CERTIFICATE_VERIFY_FAILED` contra `api.kaggle.com`.
La celda regenera el bundle combinando `certifi` con el almacén de certificados de Windows (que sí
confía en esa root). Hay que regenerarlo en cada corrida: Avast rota su root CA.

> **Ojo con el envío de `2_logit.ipynb`**: quedaron registradas dos submissions idénticas con 20 s de
> diferencia, aunque la celda corrió una sola vez (probablemente un reintento en la subida de la CLI).
> Conviene mirar el listado del final y confirmar que quedó un solo envío.

In [ ]:
import sys, os, ssl, subprocess
!{sys.executable} -m pip install --upgrade kaggle --quiet

# --- Fix SSL CERTIFICATE_VERIFY_FAILED en api.kaggle.com ---
import certifi
bundle_path = os.path.join(os.path.dirname(certifi.where()), "win_ca_bundle.pem")
with open(bundle_path, "w", encoding="utf-8") as out:
    out.write(open(certifi.where(), encoding="utf-8").read())
    ps = (
        "$s=@('Cert:\\LocalMachine\\Root','Cert:\\CurrentUser\\Root',"
        "'Cert:\\LocalMachine\\CA','Cert:\\CurrentUser\\CA');"
        "foreach($p in $s){try{Get-ChildItem $p -EA Stop|%{"
        "'-----BEGIN CERTIFICATE-----';"
        "[Convert]::ToBase64String($_.RawData,'InsertLineBreaks');"
        "'-----END CERTIFICATE-----'}}catch{}}"
    )
    win = subprocess.run(["powershell", "-NoProfile", "-Command", ps],
                         capture_output=True, text=True)
    out.write("\n" + win.stdout)

for var in ("REQUESTS_CA_BUNDLE", "SSL_CERT_FILE", "CURL_CA_BUNDLE"):
    os.environ[var] = bundle_path
print("CA bundle listo:", bundle_path)

In [ ]:
COMPETITION = "playground-series-s6e9"
FILE        = "../submissions/3_xgboost_submission.csv"
MSG         = f"XGBoost - {MEJOR} - cat nativas - CV AUC={cv_mean:.5f}"

!kaggle competitions submit -c {COMPETITION} -f "{FILE}" -m "{MSG}"

In [ ]:
# Score publico del envio (esperar unos segundos a que Kaggle lo puntue)
!kaggle competitions submissions -c playground-series-s6e9

## 9. Conclusiones

_Pendiente: se completa después de ejecutar el notebook, con los números reales a la vista._

Los puntos a cerrar cuando estén los resultados:

- CV AUC y AUC out-of-fold del mejor config, y desvío entre folds.
- Ganancia sobre el logit (0,93855) y brecha contra el tope del leaderboard (0,94631).
- Si las derivadas aportaron algo a un modelo de árboles, o si —como se anticipa arriba— resultaron
  redundantes porque XGBoost ya construye esas combinaciones.
- Qué dice el ranking de feature importance frente al AUC univariado del EDA.
- Correlación de Spearman entre los OOF del logit y de XGBoost: cuánto margen deja para un ensamble.
- Score público vs CV, y si la brecha se parece a la del logit (−0,00066).